<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Receta que Sale Bien en Casa y Mal en el Restaurante 🍳
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/00_Introduccion_Cap3_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 00 del Módulo 08](../00_Introduccion_Cap3_y_Conceptos_Avanzados.ipynb), que abre el **Capítulo 3** del curso. Hasta ahora los modelos se probaron con datos «de laboratorio»: ordenados, limpios y siempre iguales. Aquí vemos **qué pasa cuando el modelo sale al mundo real**, con cuatro historias cotidianas y mini-experimentos (solo *NumPy* y gráficas).

Al terminar podrás explicar:
1. Por qué un modelo puede **funcionar bien en las pruebas y mal en la calle** cuando el mundo cambia (el vendedor de helados y el filtro de correo).
2. Por qué **datos con errores** dañan más a quien **memoriza** que a quien **promedia** (el libro con erratas).
3. Qué es un **atajo engañoso** (la vaca que se reconoce por el pasto).

> 🗺️ **El mapa del capítulo:** este Módulo 08 trae cuatro herramientas prácticas: [reutilizar lo aprendido](01_Transfer_Learning_Dummies.ipynb), [explicar las decisiones](02_Explicabilidad_Dummies.ipynb), [clases raras](03_Desbalanceados_Dummies.ipynb) y [datos que faltan](04_Datos_Faltantes_Dummies.ipynb). Después viene el Módulo 09 (casos de estudio, imágenes, lenguaje y recomendaciones). Si luego quieres más detalle: el [cuaderno estándar](../00_Introduccion_Cap3_y_Conceptos_Avanzados.ipynb).

---
## 1. El vendedor de helados que viajó a otra estación 🍦

Imagina que aprendes a predecir las ventas de helado **observando solo días entre 15 °C y 30 °C**. En ese rango, «más calor, más helados» parece una **línea recta**. Pero en la realidad, cuando hace **muchísimo** calor la gente **sale menos** y las ventas **bajan**. Tu regla funcionó en lo que viste y **falla donde nunca estuviste**. Los expertos llaman a esto que **cambie el mundo** (que los datos nuevos sean distintos de los de entrenamiento).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)

def ventas_reales(t):
    """La 'verdad' del negocio: sube con el calor, pero cae cuando hace DEMASIADO calor."""
    return 100 / (1 + np.exp(-(t - 22) / 3)) - 0.8 * np.maximum(t - 33, 0) ** 2

t_entrenamiento = rng.uniform(15, 30, 200)                                   # solo vimos días de 15 a 30 grados
ventas = ventas_reales(t_entrenamiento) + rng.normal(0, 4, 200)
recta = np.polyfit(t_entrenamiento, ventas, 1)                               # la 'regla' aprendida: una línea recta

t_todo = np.linspace(15, 42, 100)
error = np.abs(np.polyval(recta, t_todo) - ventas_reales(t_todo))
error_conocido = error[t_todo <= 30].mean()
error_nuevo = error[t_todo > 35].mean()

fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.plot(t_todo, ventas_reales(t_todo), color="#16a34a", lw=3, label="La realidad")
ax.plot(t_todo, np.polyval(recta, t_todo), color="#dc2626", lw=2, ls="--", label="Nuestra regla (aprendida entre 15 y 30 °C)")
ax.scatter(t_entrenamiento, ventas, s=8, color="#2563eb", alpha=0.5, label="Días que vimos")
ax.axvspan(30, 42, color="#fef3c7", alpha=0.6, label="Territorio que nunca vimos")
ax.set_xlabel("Temperatura (°C)"); ax.set_ylabel("Helados vendidos"); ax.legend(fontsize=8)
ax.set_title("Una regla buena en lo conocido puede fallar en lo nuevo", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Error medio entre 15 y 30 °C (lo que vimos): {error_conocido:.1f} helados | por encima de 35 °C (lo nuevo): {error_nuevo:.1f} helados")
assert error_nuevo > 5 * error_conocido

---
---
**Lección:** nada «se rompió» en el modelo: simplemente **el mundo se movió** a un sitio donde nunca aprendió. La solución no es magia: **vigilar** si los datos nuevos se parecen a los de entrenamiento y **volver a entrenar** cuando dejan de parecerse.

## 2. El filtro de correo que envejece 📧

Un filtro marca como *spam* los correos con **muchas palabras sospechosas** (más de, digamos, 5). Pero los spammers **aprenden** y cada mes **suavizan** sus mensajes: lo que hoy es spam tiene menos palabras sospechosas. El filtro **congelado** sigue usando la regla de siempre. Otro filtro **se reentrena cada mes** con los correos del mes anterior. ¿Cómo les va a los dos?

In [ ]:
rng = np.random.default_rng(2)
meses = 12

def correos_del_mes(m, n=600):
    """Cada mes el 'umbral real' de spam baja: lo que es spam tiene cada vez menos palabras sospechosas."""
    palabras = rng.normal(5, 2.5, n)
    es_spam = (palabras > 5 - 0.4 * m).astype(int)
    ruido = rng.random(n) < 0.03                                              # 3 % de correos mal etiquetados
    return palabras, np.where(ruido, 1 - es_spam, es_spam)

def mejor_umbral(palabras, es_spam):
    candidatos = np.linspace(-2, 12, 141)
    aciertos = [np.mean((palabras > c) == es_spam) for c in candidatos]
    return candidatos[int(np.argmax(aciertos))]

datos = [correos_del_mes(m) for m in range(meses)]
umbral_fijo = mejor_umbral(*datos[0])                                          # el filtro 'congelado' aprendió en el mes 0
acc_fijo, acc_reentrenado = [], []
for m in range(1, meses):
    p, s = datos[m]
    acc_fijo.append(np.mean((p > umbral_fijo) == s))
    acc_reentrenado.append(np.mean((p > mejor_umbral(*datos[m - 1])) == s))     # reentrenado con los correos del mes anterior

fig, ax = plt.subplots(figsize=(8.5, 4.3))
ax.plot(range(1, meses), acc_fijo, "o-", color="#dc2626", lw=2, label="Filtro congelado (aprendió en el mes 0)")
ax.plot(range(1, meses), acc_reentrenado, "s-", color="#16a34a", lw=2, label="Filtro que se reentrena cada mes")
ax.set_xlabel("Mes"); ax.set_ylabel("Fracción de correos bien clasificados"); ax.legend(fontsize=8); ax.set_ylim(0.5, 1.02)
ax.set_title("Cuando las reglas del mundo cambian, hay que actualizar", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print(f"Filtro congelado: pasa de {acc_fijo[0]:.0%} (mes 1) a {acc_fijo[-1]:.0%} (mes {meses - 1}). Filtro reentrenado: se mantiene alrededor del {np.mean(acc_reentrenado):.0%}.")
assert acc_fijo[-1] < acc_fijo[0] - 0.15 and np.mean(acc_reentrenado) > 0.9

---
---
**Lección:** el filtro congelado pierde puntería **poco a poco** (y nadie recibe una alarma). Hay que **medir su desempeño con datos recientes** y **reentrenar**. Ojo: si solo se hubiera vigilado *cuántas palabras sospechosas hay en promedio*, **no se habría notado nada** (esa cifra no cambió): lo que cambió fue **qué significa** «sospechoso». Por eso hace falta **comprobar los aciertos**, no solo mirar los datos de entrada.

## 3. El libro con erratas 📖

Dos estudiantes aprenden de un libro de ejercicios donde **algunas respuestas están mal impresas**. La **memoriosa** se aprende **cada respuesta al pie de la letra**, errores incluidos. La **sensata** mira **a quince ejercicios parecidos** y se queda con la respuesta que **más se repite** (así, un error aislado se diluye). ¿Quién sufre más cuando sube la cantidad de erratas?

In [ ]:
rng = np.random.default_rng(3)

def ejercicios(n):
    y = rng.integers(0, 2, n)
    X = rng.normal(0, 1.2, (n, 2)) + np.where(y[:, None] == 1, 1.0, -1.0)      # dos 'tipos' de ejercicio, bastante distinguibles
    return X, y

def predecir_vecinos(X_ent, y_ent, X_nuevo, k):
    """Cada ejercicio nuevo se resuelve mirando a sus k ejercicios más parecidos y votando."""
    d = ((X_nuevo[:, None, :] - X_ent[None, :, :]) ** 2).sum(axis=2)
    cercanos = np.argsort(d, axis=1)[:, :k]
    return (y_ent[cercanos].mean(axis=1) > 0.5).astype(int)

X_ent, y_ent = ejercicios(300)
X_ex, y_ex = ejercicios(3000)                                                   # examen con respuestas CORRECTAS
erratas = [0.0, 0.1, 0.2, 0.3, 0.4]
res = {"Memoriosa (copia 1 ejercicio)": [], "Sensata (vota entre 15)": []}
for tasa in erratas:
    y_mal = np.where(rng.random(len(y_ent)) < tasa, 1 - y_ent, y_ent)           # el libro tiene erratas
    res["Memoriosa (copia 1 ejercicio)"].append(np.mean(predecir_vecinos(X_ent, y_mal, X_ex, 1) == y_ex))
    res["Sensata (vota entre 15)"].append(np.mean(predecir_vecinos(X_ent, y_mal, X_ex, 15) == y_ex))

fig, ax = plt.subplots(figsize=(8.5, 4.4))
for (nombre, v), c in zip(res.items(), ["#dc2626", "#16a34a"]):
    ax.plot([100 * t for t in erratas], v, "o-", color=c, lw=2, label=nombre)
ax.axhline(0.5, color="black", ls=":", label="Adivinar con una moneda")
ax.set_xlabel("% de respuestas mal impresas en el libro"); ax.set_ylabel("Aciertos en un examen con respuestas correctas"); ax.legend(fontsize=8)
ax.set_title("Memorizar también memoriza los errores", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
print({n: [f"{x:.0%}" for x in v] for n, v in res.items()})
assert res["Sensata (vota entre 15)"][2] > res["Memoriosa (copia 1 ejercicio)"][2] + 0.08

---
---
**Lección:** con erratas, **quien memoriza todo copia también los errores** y pierde mucho; **quien promedia varios casos** los diluye y resiste mejor. Moraleja práctica: **revisa la calidad de las respuestas** (las etiquetas) antes de culpar al modelo. Y cuando casi la mitad del libro miente, **nadie** puede aprender bien.

## 4. La vaca que se reconoce por el pasto 🐄

Un sistema aprende a distinguir **vacas de camellos**. En sus fotos de entrenamiento, **casi siempre hay pasto con las vacas y arena con los camellos**. Reconocer la forma del animal es **difícil** (la foto es borrosa); mirar el **fondo** es **facilísimo**. ¿Qué aprende el sistema? A mirar el fondo. Ahora le mostramos una vaca en la playa…

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(4)

def fotos(n, coincidencia):
    """forma: pista REAL pero borrosa. fondo: pista FACIL; coincide con el animal con probabilidad `coincidencia` (0 = pasto→vaca, 1 = arena→camello)."""
    animal = rng.integers(0, 2, n)                                             # 0 = camello, 1 = vaca
    forma = animal + rng.normal(0, 1.5, n)
    fondo = np.where(rng.random(n) < coincidencia, animal, 1 - animal) + rng.normal(0, 0.1, n)
    return np.column_stack([forma, fondo]), animal

X_ent, y_ent = fotos(2000, 0.95)                                               # en entrenamiento, el pasto casi siempre acompaña a las vacas
modelo = LogisticRegression().fit(X_ent, y_ent)
solo_forma = LogisticRegression().fit(X_ent[:, :1], y_ent)
casos = {"Como en entrenamiento (95 %)": 0.95, "Sin relación (50 %)": 0.5, "Vacas en la playa, camellos en el pasto (5 %)": 0.05}
filas = []
for nombre, c in casos.items():
    X, y = fotos(5000, c)
    filas.append((nombre, modelo.score(X, y), solo_forma.score(X[:, :1], y)))

fig, ax = plt.subplots(figsize=(9, 4.4))
pos = np.arange(len(filas))
ax.bar(pos - 0.18, [f[1] for f in filas], 0.36, color="#dc2626", label="Mira forma y fondo (usa el atajo)")
ax.bar(pos + 0.18, [f[2] for f in filas], 0.36, color="#16a34a", label="Mira solo la forma")
ax.axhline(0.5, color="black", ls=":"); ax.set_xticks(pos); ax.set_xticklabels([f[0].replace(" (", "\n(") for f in filas], fontsize=8)
ax.set_ylabel("Aciertos"); ax.set_ylim(0, 1.05); ax.legend(fontsize=8)
ax.set_title("El atajo gana en la práctica de casa y pierde en el mundo real", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
for nombre, a, b in filas:
    print(f"{nombre}: con atajo {a:.0%} | solo la forma {b:.0%}")
assert filas[0][1] > 0.93 and filas[2][1] < 0.15 and abs(filas[0][2] - filas[2][2]) < 0.03

---
---
**Lección:** el sistema con el atajo saca **~95 %** en las pruebas «de casa» y **cae a ~5 %** cuando el fondo engaña; el que mira **solo la forma** saca menos (~63 %) pero **siempre lo mismo**: es **honesto y robusto**. **Un buen resultado en las pruebas no garantiza que el modelo haya aprendido lo importante.** Para descubrirlo hace falta **probar en situaciones distintas** y usar el **sentido común** (¿tiene sentido que decida por el fondo?). Predecir bien **no es lo mismo que entender las causas**.

## Resumen en una frase 🎯

> **Un modelo es como una receta: puede salir perfecta en tu cocina y fallar en otro restaurante; hay que probarla en condiciones nuevas, vigilarla cuando cambie el mundo, revisar los datos con errores y desconfiar de los atajos.**

### Lo que aprendiste hoy:
- ✅ **El mundo cambia** (el vendedor de helados): una regla aprendida en un rango falla fuera de él.
- ✅ **Las reglas envejecen** (el filtro de correo): hay que **comprobar los aciertos** con datos recientes y **reentrenar**.
- ✅ **Los errores en los datos** (el libro con erratas) dañan más a quien **memoriza** que a quien **promedia**.
- ✅ **Los atajos** (la vaca y el pasto) ganan en el laboratorio y fallan en la calle: **buen resultado ≠ aprendió lo correcto**.

> 🎓 **Siguiente paso:** el [cuaderno estándar](../00_Introduccion_Cap3_y_Conceptos_Avanzados.ipynb) mide todo esto con rigor y cubre además cómo **combinar modelos** (*stacking*); y luego, [cómo reutilizar lo ya aprendido (transfer learning)](01_Transfer_Learning_Dummies.ipynb).

---
##### 🛠️ Práctica 1: Ponle una alarma al filtro de correo

Al filtro **congelado** nadie le avisó cuando empezó a fallar. Vamos a ponerle una **alarma**: «si acierta menos del **85 %**, hay que reentrenar». Usa la lista `acc_fijo` (los aciertos del filtro congelado en los meses 1, 2, 3...), `meses`, `datos` (los correos de cada mes) y la función `mejor_umbral`:

1. Busca en qué mes **suena la alarma**: el primer mes en que el acierto baja de `0.85`. Guárdalo en `mes_alarma`.
2. **Reentrena** el filtro con los correos del mes anterior: `umbral_nuevo = mejor_umbral(*datos[mes_alarma - 1])`.
3. **Pruébalo** con los correos de ese mes (`datos[mes_alarma]`) y guarda su acierto en `acc_nuevo`.
4. Comprueba con `assert` que el filtro reentrenado supera el 90 % y que mejora al congelado en ese mes.

In [ ]:
# Escribe tu código aquí

# 1) ¿En qué mes baja el acierto del filtro congelado de 0.85?
# mes_alarma = next(m for m, a in zip(range(1, meses), acc_fijo) if a < ...)

# 2) Reentrena con los correos del mes anterior
# umbral_nuevo = mejor_umbral(*datos[...])

# 3) Pruébalo con los correos de ese mes
# palabras, es_spam = datos[mes_alarma]
# acc_nuevo = np.mean((palabras > ...) == es_spam)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# 1) Primer mes en que el filtro congelado cae por debajo del 85 %
mes_alarma = next(m for m, a in zip(range(1, meses), acc_fijo) if a < 0.85)
print(f"La alarma suena en el mes {mes_alarma}: el filtro congelado acierta {acc_fijo[mes_alarma - 1]:.0%}")

# 2) Reentrenar = volver a buscar el mejor umbral con los correos del mes anterior
umbral_nuevo = mejor_umbral(*datos[mes_alarma - 1])

# 3) Probarlo con los correos de ese mes
palabras, es_spam = datos[mes_alarma]
acc_nuevo = np.mean((palabras > umbral_nuevo) == es_spam)
print(f"Umbral viejo = {umbral_fijo:.1f} palabras | umbral nuevo = {umbral_nuevo:.1f} | acierto reentrenado = {acc_nuevo:.0%}")

assert acc_nuevo > 0.9 and acc_nuevo > acc_fijo[mes_alarma - 1] + 0.05
```
</details>

---


---
##### 🛠️ Práctica 2: Probar a la vaca en cinco situaciones

Un solo examen no basta para descubrir un atajo. Prueba a los dos sistemas (`modelo`, que mira forma y fondo, y `solo_forma`, que mira únicamente la forma) en **cinco situaciones** distintas usando la función `fotos(n, coincidencia)`:

1. Haz la lista `coincidencias = [0.95, 0.8, 0.5, 0.2, 0.05]` (qué tan seguido el fondo «acierta» con el animal).
2. Para cada valor, genera `X, y = fotos(5000, c)` y guarda el acierto de `modelo` (con `X`) en la lista `acc_atajo` y el de `solo_forma` (con `X[:, :1]`) en la lista `acc_forma`.
3. Comprueba con `assert` que `acc_atajo` **baja** en cada situación, que termina por debajo del 10 %, y que `acc_forma` casi no se mueve (menos de 5 puntos entre su mejor y su peor valor).

In [ ]:
# Escribe tu código aquí

# coincidencias = [0.95, 0.8, 0.5, 0.2, 0.05]
# acc_atajo, acc_forma = [], []
# for c in coincidencias:
#     X, y = fotos(5000, c)
#     acc_atajo.append(modelo.score(X, y))
#     acc_forma.append(solo_forma.score(X[:, :1], y))


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
coincidencias = [0.95, 0.8, 0.5, 0.2, 0.05]
acc_atajo, acc_forma = [], []
for c in coincidencias:
    X, y = fotos(5000, c)                                   # fotos nuevas donde el fondo acierta con probabilidad c
    acc_atajo.append(modelo.score(X, y))                    # mira forma y fondo
    acc_forma.append(solo_forma.score(X[:, :1], y))         # mira solo la forma
    print(f"fondo acierta {c:.0%}: con atajo {acc_atajo[-1]:.0%} | solo forma {acc_forma[-1]:.0%}")

assert all(a > b for a, b in zip(acc_atajo, acc_atajo[1:])) and acc_atajo[-1] < 0.1
assert max(acc_forma) - min(acc_forma) < 0.05
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué una regla aprendida entre 15 y 30 grados puede fallar a 40 grados? Da otro ejemplo de tu vida (una receta, una ruta, un horario).
2. Tu filtro de correo sigue diciendo que todo está bien, pero cada semana pasan más correos basura. ¿Qué comprobarías primero y qué harías?
3. Un amigo dice: «mi sistema de reconocer enfermedades acertó 98 % en las pruebas, ¡es perfecto!». ¿Qué le preguntarías sobre **de dónde salieron** las fotos de entrenamiento y las de prueba?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>